In [ ]:
# The purpose of this python notebook is to:
#   - understand the ATO data that Natalia sent over
#   - develop initial code used to orginize/merge/prep the data for viewing on map
#   - investigate VMT outputs as well


In [4]:
import pandas as pd
import geopandas as gpd
import os


In [5]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


## Inputs


In [6]:
bq_taz_ustm4 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_taz.ustm_v4_taz_2025_07_29_geo"
).to_dataframe()
gdf_taz = gpd.GeoDataFrame(bq_taz_ustm4)
gdf_taz_short = gdf_taz[["CO_TAZID", "geometry"]]


c:\Users\cday\anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
file_paths = {
    "path_ato_SA0_19": r"_data/raw/0 - USTM/2019/Access_to_Opportunity.csv",
    "path_ato_SA0_23": r"_data/raw/0 - USTM/2023/Access_to_Opportunity.csv",
    "path_ato_SA0_28": r"_data/raw/0 - USTM/2028/Access_to_Opportunity.csv",
    "path_ato_SA1_19": r"_data/raw/1 - WF/2023/Access_to_Opportunity.csv",
    "path_ato_SA1_23": r"_data/raw/1 - WF/2028/Access_to_Opportunity.csv",
    "path_ato_SA1_28": r"_data/raw/1 - WF/2019/Access_to_Opportunity.csv",
    "path_ato_SA2_23": r"_data/raw/2 - CA/2023/Access_to_Opportunity.csv",
    "path_ato_SA2_28": r"_data/raw/2 - CA/2028/Access_to_Opportunity.csv",
    "path_ato_SA3_19": r"_data/raw/3 - DX/2019/Access_to_Opportunity.csv",
    "path_ato_SA3_23": r"_data/raw/3 - DX/2023/Access_to_Opportunity.csv",
    "path_ato_SA3_28": r"_data/raw/3 - DX/2028/Access_to_Opportunity.csv",
    "path_ato_SA4_19": r"_data/raw/4 - WB/2019/Access_to_Opportunity.csv",
    "path_ato_SA4_23": r"_data/raw/4 - WB/2023/Access_to_Opportunity.csv",
    "path_ato_SA4_28": r"_data/raw/4 - WB/2028/Access_to_Opportunity.csv",
    "path_ato_SA5_19": r"_data/raw/5 - IR/2019/Access_to_Opportunity.csv",
    "path_ato_SA5_23": r"_data/raw/5 - IR/2023/Access_to_Opportunity.csv",
    "path_ato_SA5_28": r"_data/raw/5 - IR/2028/Access_to_Opportunity.csv",
}

ustm_missing_cols = {
    "HH_byTran",
    "Job_byTran",
    "act_HH_byTran",
    "act_Job_byTran",
    "tmp_HH_byTran_dr",
    "tmp_HH_byTran_wk",
    "tmp_Job_byTran_dr",
    "tmp_Job_byTran_wk",
}

rename_cols = {"TAZID": "SA_TAZID"}

metadata_map = {
    "path_ato_SA0_19": {"ModelArea": "Statewide", "ScenarioYear": 2019},
    "path_ato_SA0_23": {"ModelArea": "Statewide", "ScenarioYear": 2023},
    "path_ato_SA0_28": {"ModelArea": "Statewide", "ScenarioYear": 2028},
    "path_ato_SA1_19": {"ModelArea": "Wasatch Front", "ScenarioYear": 2019},
    "path_ato_SA1_23": {"ModelArea": "Wasatch Front", "ScenarioYear": 2023},
    "path_ato_SA1_28": {"ModelArea": "Wasatch Front", "ScenarioYear": 2028},
    "path_ato_SA2_23": {"ModelArea": "Cache", "ScenarioYear": 2023},
    "path_ato_SA2_28": {"ModelArea": "Cache", "ScenarioYear": 2028},
    "path_ato_SA3_19": {"ModelArea": "Dixie", "ScenarioYear": 2019},
    "path_ato_SA3_23": {"ModelArea": "Dixie", "ScenarioYear": 2023},
    "path_ato_SA3_28": {"ModelArea": "Dixie", "ScenarioYear": 2028},
    "path_ato_SA4_19": {"ModelArea": "Summit Wasatch", "ScenarioYear": 2019},
    "path_ato_SA4_23": {"ModelArea": "Summit Wasatch", "ScenarioYear": 2023},
    "path_ato_SA4_28": {"ModelArea": "Summit Wasatch", "ScenarioYear": 2028},
    "path_ato_SA5_19": {"ModelArea": "Iron", "ScenarioYear": 2019},
    "path_ato_SA5_23": {"ModelArea": "Iron", "ScenarioYear": 2023},
    "path_ato_SA5_28": {"ModelArea": "Iron", "ScenarioYear": 2028},
}

need_columns = [
    "ScenarioYear",
    "CO_TAZID",
    "ModelArea",
]
base_columns = [
    "Job_byAuto",
    "Job_byTran",
    "Job_byBike",
    "Job_byWalk",
    "HH_byAuto",
    "HH_byTran",
    "HH_byBike",
    "HH_byWalk",
]


## ATO Data


In [8]:
# Process each file and store the resulting dataframes in a list
df_list = []
for key, path in file_paths.items():
    # read in, rename, add cols
    df = pd.read_csv(path)
    df = df.rename(columns=rename_cols)
    df["ModelArea"] = metadata_map[key]["ModelArea"]
    df["ScenarioYear"] = metadata_map[key]["ScenarioYear"]

    # Add missing USTM columns
    if "SA0" in key:
        for col in ustm_missing_cols:
            df[col] = 0

    df_list.append(df)

# Concatenate all the dataframes together
df_combined_ato = pd.concat(df_list, ignore_index=True)

# reorganize columns
first_cols = ["ScenarioYear", "CO_TAZID", "ModelArea", "SA_TAZID"]
remaining_cols = [col for col in df_combined_ato.columns if col not in first_cols]
df_combined_ato = df_combined_ato[first_cols + remaining_cols]

df_combined_ato


,ScenarioYear,CO_TAZID,ModelArea,SA_TAZID,DevAcres,HH,Job,Job_byAuto,Job_byBike,Job_byWalk,...,tmp_HH_byAuto_FF,tmp_HH_byAuto_SL,tmp_Job_byTran_wk,act_Job_byTran,tmp_HH_byTran_wk,HH_byTran,Job_byTran,tmp_Job_byTran_dr,act_HH_byTran,tmp_HH_byTran_dr
0,2019,1001,Statewide,41,107.48750,0.0,139.9,2637,2290,636,...,1800,3048,0,0,0,0,0,0,0,0
1,2019,1002,Statewide,42,345.74170,24.0,152.5,2567,2216,287,...,1661,2791,0,0,0,0,0,0,0,0
2,2019,1003,Statewide,43,440.68060,37.4,45.9,2570,2186,76,...,1669,2845,0,0,0,0,0,0,0,0
3,2019,1004,Statewide,44,306.11370,4.8,0.0,2472,2122,20,...,1499,2519,0,0,0,0,0,0,0,0
4,2019,1005,Statewide,45,245.42460,76.7,654.3,2599,2346,908,...,1731,2863,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47199,2028,21356,Iron,356,1798.73167,8.0,0.0,22073,462,0,...,17368,20475,0,0,0,0,0,0,0,0
47200,2028,21357,Iron,357,6283.55925,0.0,0.0,23743,3185,0,...,18468,21324,0,0,0,0,0,0,0,0
47201,2028,21358,Iron,358,38092.04650,6.9,27.2,15029,27,27,...,10184,17721,0,0,0,0,0,0,0,0
47202,2028,21359,Iron,359,6043.87588,0.0,0.0,13963,14,0,...,9368,16471,0,0,0,0,0,0,0,0


In [9]:
df_combined_ato_v1 = df_combined_ato.copy()
df_combined_ato_v1 = df_combined_ato_v1[need_columns + base_columns]

gdf_combinded_ato_v1 = df_combined_ato_v1.merge(
    gdf_taz_short, how="left", on="CO_TAZID"
)
gdf_combinded_ato_v1["geometry"] = gpd.GeoSeries.from_wkt(
    gdf_combinded_ato_v1["geometry"]
)
gdf_combinded_ato_v1 = gpd.GeoDataFrame(gdf_combinded_ato_v1, geometry="geometry")
gdf_combinded_ato_v1


,ScenarioYear,CO_TAZID,ModelArea,Job_byAuto,Job_byTran,Job_byBike,Job_byWalk,HH_byAuto,HH_byTran,HH_byBike,HH_byWalk,geometry
0,2019,1001,Statewide,2637,0,2290,636,1800,0,1203,61,"POLYGON ((-112.65114 38.29355, -112.65117 38.2..."
1,2019,1002,Statewide,2567,0,2216,287,1660,0,1194,38,"POLYGON ((-112.64859 38.30477, -112.65288 38.3..."
2,2019,1003,Statewide,2570,0,2186,76,1668,0,1212,44,"POLYGON ((-112.62542 38.30818, -112.62872 38.3..."
3,2019,1004,Statewide,2472,0,2122,20,1498,0,1195,22,"POLYGON ((-112.62394 38.30818, -112.62535 38.3..."
4,2019,1005,Statewide,2599,0,2346,908,1731,0,1246,179,"POLYGON ((-112.64491 38.29335, -112.64513 38.2..."
...,...,...,...,...,...,...,...,...,...,...,...,...
47199,2028,21356,Iron,22073,0,462,0,17239,0,718,8,"POLYGON ((-112.78001 37.87921, -112.77996 37.8..."
47200,2028,21357,Iron,23743,0,3185,0,18358,0,2046,0,"POLYGON ((-112.77254 38.03363, -112.77259 38.0..."
47201,2028,21358,Iron,15029,0,27,27,10022,0,7,7,"POLYGON ((-112.77358 37.89081, -112.77358 37.8..."
47202,2028,21359,Iron,13963,0,14,0,9209,0,1,0,"POLYGON ((-112.76553 37.89698, -112.76553 37.8..."


In [10]:
# interactive map code here!
import numpy as np
import pandas as pd
import ipywidgets as widgets
import folium
import branca.colormap as cm
from IPython.display import display
from folium.features import GeoJsonTooltip

_access_gdf = gdf_combinded_ato_v1.copy()
_access_gdf = _access_gdf.dropna(subset=["geometry"]).copy()
_access_gdf = _access_gdf[~_access_gdf.geometry.is_empty].copy()

if _access_gdf.crs is None:
    _access_gdf = _access_gdf.set_crs("EPSG:4326", allow_override=True)
else:
    _access_gdf = _access_gdf.to_crs("EPSG:4326")

_model_area_order = [
    "Statewide",
    "Wasatch Front",
    "Cache",
    "Dixie",
    "Summit Wasatch",
    "Iron",
]


def _ordered_model_areas(values):
    values = list(values)
    ordered = [area for area in _model_area_order if area in values]
    ordered.extend(sorted(area for area in values if area not in ordered))
    return ordered


_year_options = sorted(_access_gdf["ScenarioYear"].dropna().unique().tolist())
_areas_by_year = {
    year: _ordered_model_areas(
        _access_gdf.loc[_access_gdf["ScenarioYear"].eq(year), "ModelArea"]
        .dropna()
        .unique()
        .tolist()
    )
    for year in _year_options
}

year_widget = widgets.Dropdown(
    options=_year_options,
    value=_year_options[0],
    description="Year:",
    layout=widgets.Layout(width="180px"),
)
area_widget = widgets.Dropdown(
    options=_areas_by_year[year_widget.value],
    value=_areas_by_year[year_widget.value][0],
    description="Area:",
    layout=widgets.Layout(width="260px"),
)
access_widget = widgets.ToggleButtons(
    options=[("Jobs", "Job"), ("HH", "HH")],
    value="Job",
    description="Access:",
)
mode_widget = widgets.ToggleButtons(
    options=[("Auto", "Auto"), ("Transit", "Tran"), ("Walk", "Walk"), ("Bike", "Bike")],
    value="Auto",
    description="Mode:",
)


def _update_area_options(change):
    options = _areas_by_year.get(change["new"], [])
    current = area_widget.value
    area_widget.options = options
    if options:
        area_widget.value = current if current in options else options[0]


year_widget.observe(_update_area_options, names="value")


def _make_access_map(scenario_year, model_area, access_to, mode):
    value_col = f"{access_to}_by{mode}"
    access_label = "jobs" if access_to == "Job" else "households"
    mode_label = {"Auto": "auto", "Tran": "transit", "Walk": "walk", "Bike": "bike"}[
        mode
    ]

    selected_gdf = _access_gdf.loc[
        _access_gdf["ScenarioYear"].eq(scenario_year)
        & _access_gdf["ModelArea"].eq(model_area),
        ["ScenarioYear", "ModelArea", "CO_TAZID", value_col, "geometry"],
    ].copy()
    selected_gdf[value_col] = pd.to_numeric(selected_gdf[value_col], errors="coerce")
    selected_gdf = selected_gdf.dropna(subset=[value_col, "geometry"])
    selected_gdf = selected_gdf[~selected_gdf.geometry.is_empty].copy()

    if selected_gdf.empty:
        display(
            widgets.HTML(
                f"<b>No TAZ records found for {scenario_year}, {model_area}, {value_col}.</b>"
            )
        )
        return

    values = selected_gdf[value_col]
    value_min = float(values.min())
    value_max = float(values.max())
    color_min = value_min
    color_max = value_max
    if value_min == value_max:
        color_min = value_min - 1 if value_min != 0 else 0
        color_max = value_max + 1

    colormap = cm.LinearColormap(
        colors=["#f7fcf0", "#ccebc5", "#7bccc4", "#2b8cbe", "#084081"],
        vmin=color_min,
        vmax=color_max,
        caption=f"{value_col}: least accessible to most accessible",
    )

    selected_gdf["AccessValue"] = selected_gdf[value_col].map(
        lambda value: f"{value:,.0f}"
    )
    bounds = selected_gdf.total_bounds
    if not np.isfinite(bounds).all():
        display(
            widgets.HTML(
                f"<b>Could not map {scenario_year}, {model_area}: invalid geometry bounds.</b>"
            )
        )
        return

    center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
    access_map = folium.Map(
        location=center,
        zoom_start=8,
        tiles="cartodbpositron",
        control_scale=True,
    )

    def _style_function(feature):
        value = feature["properties"].get(value_col)
        return {
            "fillColor": colormap(value),
            "color": "#404040",
            "weight": 0.25,
            "fillOpacity": 0.78,
        }

    folium.GeoJson(
        selected_gdf,
        name=f"{model_area} TAZs",
        style_function=_style_function,
        highlight_function=lambda feature: {
            "color": "#111111",
            "weight": 1.5,
            "fillOpacity": 0.9,
        },
        tooltip=GeoJsonTooltip(
            fields=["CO_TAZID", "AccessValue"],
            aliases=["CO_TAZID", f"{value_col}"],
            sticky=False,
        ),
        smooth_factor=0.5,
    ).add_to(access_map)

    colormap.add_to(access_map)
    access_map.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

    display(
        widgets.HTML(
            f"<b>{model_area} {scenario_year}</b> | "
            f"Access to {access_label} by {mode_label}: "
            f"{len(selected_gdf):,} TAZs, min {value_min:,.0f}, max {value_max:,.0f}"
        )
    )
    display(access_map)


map_output = widgets.interactive_output(
    _make_access_map,
    {
        "scenario_year": year_widget,
        "model_area": area_widget,
        "access_to": access_widget,
        "mode": mode_widget,
    },
)

display(
    widgets.VBox(
        [
            widgets.HBox([year_widget, area_widget]),
            widgets.HBox([access_widget, mode_widget]),
            map_output,
        ]
    )
)


In [ ]:
# STEPS
# merge geography
# create local python map view with widgets to "double check" output looks okay
# start creating APP to view ATO
#   - create style guide from Housing Site App
#   - have it identify the "inputs" needed and the format. Then code this up in python to spit it out
#   - create the app!
# create uv environment
# convert python notebook to static .py file (data input --> map input)
# update readme to explain how to build using uv

# NOTES
#   - the USTM data has ATO for the whole state! That's really all we want. However, I included all the subareas in case we want to do a "Zoom In"
#     level comparison where we only look at the ATO of that subregion alone. I think the webapp should have both capabilities callable



## Process Parquet Data


In [ ]:
# process all needed inputs for map into parque files here
